In [ ]:
import os
import hashlib
import requests
from tqdm import tqdm
import obonet

In [ ]:
CONFIG = {
    "release_date": "2025-06-01",
    "base_url": "https://release.geneontology.org",
    "filename": "go-basic.obo",
    "output_dir": "./input_data/go",
    "chunk_size": 1024 * 1024,  # 1MB
}

In [ ]:
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def sha256_file(path):
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

In [ ]:
def download_go_basic():
    url = f"{CONFIG['base_url']}/{CONFIG['release_date']}/ontology/{CONFIG['filename']}"
    
    output_path = os.path.join(
        CONFIG["output_dir"],
        CONFIG["release_date"],
        CONFIG["filename"]
    )
    
    ensure_dir(os.path.dirname(output_path))

    if os.path.exists(output_path):
        print(f"[INFO] File already exists: {output_path}")
        return output_path

    print(f"[INFO] Downloading from: {url}")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(output_path, "wb") as f, tqdm(
        total=total_size,
        unit="B",
        unit_scale=True
    ) as pbar:
        for chunk in response.iter_content(CONFIG["chunk_size"]):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

    print(f"[INFO] Download completed: {output_path}")
    return output_path

In [ ]:
def validate_go_file(path):
    print("\n[INFO] Validating GO file...")

    # Size check
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"[INFO] File size: {size_mb:.2f} MB")

    if size_mb < 20 or size_mb > 50:
        raise ValueError("File size out of expected range!")

    # Header check
    with open(path, "r", encoding="utf-8") as f:
        for _ in range(50):
            line = f.readline()
            if "data-version" in line:
                print(f"[INFO] Found version line: {line.strip()}")
                if CONFIG["release_date"] not in line:
                    raise ValueError("Release date mismatch!")
                break

    print("[INFO] Validation passed.")

In [ ]:
def parse_go_basic(path):
    print("\n[INFO] Parsing GO file...")

    graph = obonet.read_obo(path)
    print(f"[INFO] Nodes: {len(graph.nodes)}")
    print(f"[INFO] Edges: {len(graph.edges)}")
    return {
        "num_terms": len(graph.nodes),
        "num_edges": len(graph.edges),
        "terms": set(graph.nodes)
    }

In [ ]:
def compare_go_files(path1, path2):
    print("\n[INFO] Comparing files...")

    # Hash comparison
    hash1 = sha256_file(path1)
    hash2 = sha256_file(path2)

    print(f"[INFO] Hash1: {hash1}")
    print(f"[INFO] Hash2: {hash2}")

    if hash1 == hash2:
        print("[SUCCESS] Files are IDENTICAL (byte-level)")
        return True

    print("[WARN] Files differ at binary level → doing structural compare...")

    # Structural comparison
    parsed1 = parse_go_basic(path1)
    parsed2 = parse_go_basic(path2)

    print("\n[INFO] Structural comparison:")

    print(f"Terms count: {parsed1['num_terms']} vs {parsed2['num_terms']}")
    print(f"Edges count: {parsed1['num_edges']} vs {parsed2['num_edges']}")

    missing_terms = parsed1["terms"] - parsed2["terms"]
    extra_terms = parsed2["terms"] - parsed1["terms"]

    print(f"Missing terms: {len(missing_terms)}")
    print(f"Extra terms: {len(extra_terms)}")

    if not missing_terms and not extra_terms:
        print("[SUCCESS] Structurally identical")
        return True
    else:
        print("[FAIL] Differences detected")
        return False

In [ ]:
downloaded_path = download_go_basic()

In [ ]:
reference_file_path = "comp_data/Train/go-basic.obo"
validate_go_file(downloaded_path)
parse_go_basic(downloaded_path)
compare_go_files(downloaded_path, reference_file_path)